# 03 — Il confronto (tentativo 09)

Sulle 301 testate valutate, fold per gruppo: il **riferimento** (il TF-IDF per articolo delle Versioni 1–4, con C
scelto in validazione interna, ricalcolato da `01_riferimento_tfidf` sullo stesso test), le tre **famiglie** (testo,
link, pagina), le **combinazioni** e il modello **"alla NewsGuard"** (`02_modelli`).

Metriche per testata: F1 macro a 5 livelli e accuratezza binaria (affidabile = punteggio ≥ 60), con intervalli al 95%
da un **bootstrap per gruppo** (2000 ricampionamenti) e le differenze con il riferimento sugli stessi ricampionamenti.
Poi i segnali che pesano di più nei modelli link e pagina, e quanto bene si stimano i 9 criteri.

In [1]:
import numpy as np
import pandas as pd

from comune import CLASSI, CRITERI, RESULTS, metriche_di, per_testata

B = 2000
OUT = RESULTS / "confronto"; OUT.mkdir(parents=True, exist_ok=True)
pr = pd.read_csv(RESULTS / "modelli" / "predizioni_testate.csv").set_index("domain")
if (RESULTS / "modelli_bert" / "predizioni_testate.csv").exists():     # ModernBERT come famiglia del testo (04-05), se c'è
    pb = pd.read_csv(RESULTS / "modelli_bert" / "predizioni_testate.csv").set_index("domain")
    pr = pr.join(pb[[x for x in pb.columns if " | p_l" in x]], how="inner")
pr = pr[pr.valutazione]
modelli = sorted({c.split(" | ")[0] for c in pr.columns if " | p_l" in c})
P = {m: pr[[f"{m} | p_l{k}" for k in range(1, 6)]].to_numpy().argmax(1) for m in modelli}
rif = per_testata(pd.read_csv(RESULTS / "feature_tuned_fold_gruppo_articoli.csv").rename(columns={"y_pred": "y_pred"}))
P = {"riferimento: TF-IDF per articolo (versioni 1-4)": rif.y_pred.reindex(pr.index).to_numpy(), **P}
y = pr.y_true.to_numpy(); g = pr.index.map(pd.read_csv("../Dataset/versioni/fold_finale.csv").set_index("domain").gruppo).to_numpy()
print(f"{len(pr)} testate valutate, {len(np.unique(g))} gruppi; modelli: {list(P)}")

301 testate valutate, 218 gruppi; modelli: ['riferimento: TF-IDF per articolo (versioni 1-4)', 'ModernBERT', 'alla NewsGuard', 'combinato (ModernBERT+link+pagina)', 'combinato (link+pagina)', 'combinato (testo+ModernBERT)', 'combinato (testo+link)', 'combinato (testo+pagina)', 'combinato (tutte + ModernBERT)', 'combinato (tutte)', 'link', 'pagina', 'testo']


In [2]:
gruppi = np.unique(g); pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
camp = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(p, idx):
    m = metriche_di(y[idx], p[idx]); return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(p, i) for i in camp]) for k, p in P.items()}
R = list(P)[0]
righe = {}
for k, p in P.items():
    m = metriche_di(y, p); b = boot[k]
    r = {"F1 macro": m["f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": m["bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5),
         "accuratezza": m["accuratezza"], "tolerant 1": m["tolerant_1"]}
    if k != R:
        d = b - boot[R]
        r.update({"ΔF1 vs riferimento": m["f1_macro"] - metriche_di(y, P[R])["f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5),
                  "ΔF1 97.5%": np.percentile(d[:, 0], 97.5), "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5)})
    righe[k] = r
tab = pd.DataFrame(righe).T
tab.to_csv(OUT / "intervalli.csv")
tab.round(3)

,F1 macro,F1 2.5%,F1 97.5%,binario,bin 2.5%,bin 97.5%,accuratezza,tolerant 1,ΔF1 vs riferimento,ΔF1 2.5%,ΔF1 97.5%,Δbin 2.5%,Δbin 97.5%
riferimento: TF-IDF per articolo (versioni 1-4),0.340,0.280,0.395,0.684,0.623,0.751,0.369,0.741,NaN,NaN,NaN,NaN,NaN
ModernBERT,0.325,0.272,0.375,0.708,0.633,0.780,0.409,0.774,-0.015,-0.067,0.042,-0.020,0.071
alla NewsGuard,0.319,0.260,0.378,0.718,0.651,0.783,0.385,0.718,-0.021,-0.100,0.063,-0.028,0.095
combinato (ModernBERT+link+pagina),0.304,0.247,0.358,0.704,0.637,0.769,0.336,0.734,-0.036,-0.111,0.039,-0.032,0.070
combinato (link+pagina),0.300,0.234,0.359,0.681,0.611,0.743,0.312,0.674,-0.040,-0.121,0.041,-0.088,0.077
combinato (testo+ModernBERT),0.328,0.263,0.383,0.718,0.647,0.784,0.355,0.738,-0.012,-0.080,0.058,-0.021,0.085
combinato (testo+link),0.328,0.262,0.387,0.688,0.619,0.758,0.349,0.674,-0.012,-0.092,0.075,-0.056,0.061
combinato (testo+pagina),0.354,0.290,0.414,0.711,0.648,0.772,0.372,0.744,0.014,-0.056,0.086,-0.027,0.077
combinato (tutte + ModernBERT),0.318,0.258,0.373,0.714,0.649,0.778,0.339,0.748,-0.022,-0.097,0.054,-0.024,0.082
combinato (tutte),0.347,0.286,0.405,0.688,0.619,0.756,0.365,0.708,0.007,-0.068,0.085,-0.059,0.066


## I segnali che pesano di più (modelli link e pagina, media sui 5 fold)

In [3]:
co = pd.read_csv(RESULTS / "modelli" / "coefficienti_link_pagina.csv")
top = co.groupby(["famiglia", "livello", "segnale"]).peso.agg(["mean", "count"]).reset_index()
top = top[top["count"] >= 3].sort_values("mean", ascending=False).groupby(["famiglia", "livello"]).head(8)
top.to_csv(OUT / "segnali_piu_forti.csv", index=False)
for (fam, liv), x in top.groupby(["famiglia", "livello"]):
    print(f"{fam} -> {liv}: " + ", ".join(f"{s} ({w:+.2f})" for s, w in zip(x.segnale, x["mean"])))

link -> l1: t.me (+1.48), nbcnews.com (+1.42), tinyurl.com (+1.20), gab.com (+1.19), statista.com (+1.09), rt-english.com (+1.07), techcrunch.com (+0.91), pravda.ru (+0.85)
link -> l2: twitter.com (+1.57), thelancet.com (+1.44), dailywire.com (+1.38), paypal.com (+1.35), twc.health (+1.29), dawn.com (+1.12), sendgrid.net (+1.12), rasmussenreports.com (+1.03)
link -> l3: pxf.io (+1.62), facebook.com (+1.42), fox59.com (+1.26), pagesix.com (+1.18), history.com (+1.11), twitter.com (+1.08), www.nhs.uk (+0.99), proofpoint.com (+0.83)
link -> l4: patreon.com (+1.38), instagram.com (+1.11), publicnoticeportal.uk (+1.06), news.com.au (+0.98), sciencedirect.com (+0.81), urldefense.com (+0.74), cnn.com (+0.74), nytimes.com (+0.70)
link -> l5: apple.news (+1.11), bbc.com (+1.10), army.mil (+1.07), nine.com.au (+1.00), canada.ca (+0.99), politico.eu (+0.94), mod.uk (+0.88), apple.com (+0.87)
pagina -> l1: incorporati: video alternativi (+0.45), link nella pagina (+0.18), sezione commenti (+0.17),

## Quanto bene si stimano i criteri NewsGuard

In [4]:
cr = pd.read_csv(RESULTS / "modelli" / "criteri.csv", index_col=0)
cr.round(3)

,accuratezza bilanciata,quota sì (vera)
Does not repeatedly publish false content,0.670,0.794
Gathers and presents information responsibly,0.766,0.475
Regularly corrects or clarifies errors,0.702,0.432
Handles the difference between news and opinion responsibly,0.610,0.535
Avoids deceptive headlines,0.718,0.784
Website discloses ownership and financing,0.565,0.442
Clearly labels advertising,0.544,0.744
"Reveals who's in charge, including any possible conflicts of interest",0.621,0.681
"The site provides names of content creators, along with either contact or biographical information",0.632,0.671
